# 2. Vector Data Ingestion

This notebook takes the clustered EMR data, generates summaries, and ingests them into the Qdrant vector database.

In [1]:
import os
import sys
import pandas as pd

# Ultimate foolproof path injection to find 'src' regardless of current working directory
cwd = os.path.abspath(os.getcwd())
project_root = None

# 1. Walk up from current working directory
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent

# 2. Fallback to searching sys.path
if not project_root:
    for path in list(sys.path):
        if not path:
            continue
        candidate = os.path.abspath(path)
        for folder in [candidate, os.path.abspath(os.path.join(candidate, '..')), os.path.abspath(os.path.join(candidate, '..', '..'))]:
            if os.path.exists(os.path.join(folder, 'src', 'config.py')):
                project_root = folder
                break
        if project_root:
            break

if project_root:
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    path_prefix = os.path.relpath(project_root, cwd)
    print(f"Project root found: {project_root}")
    print(f"Current working directory: {cwd}")
    print(f"Relative path prefix: {path_prefix}")
else:
    print("Warning: Could not automatically detect project root. Using default path prefix '..'")
    path_prefix = ".."

from src.config import settings
from src.transform import get_model_summaries, get_site_summaries
from src.utils import get_embeddings

from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore

Project root found: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator
Current working directory: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\notebook
Relative path prefix: ..


d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("1. Loading Clustered Data...")
file_path = os.path.join(path_prefix, "output", "clustered_emr.csv")
if not os.path.exists(file_path):
    raise FileNotFoundError("clustered_emr.csv not found. Run 1_clustering.ipynb first.")

df = pd.read_csv(file_path)
print(f"Loaded {len(df)} clustered records.")

1. Loading Clustered Data...
Loaded 20630 clustered records.


In [3]:
print("2. Creating LangChain Documents...")
docs = []

if 'combined_text' not in df.columns:
    print("  'combined_text' column not found in CSV. Generating rich semantic text using transform_emr_row_to_text...")
    from src.transform import transform_emr_row_to_text
    df['combined_text'] = df.apply(transform_emr_row_to_text, axis=1)

for _, row in df.iterrows():
    content = row['combined_text']
    # Remove the combined_text from metadata to avoid duplication
    metadata = {k: str(v) for k, v in row.items() if k != 'combined_text' and pd.notna(v)}
    docs.append(Document(page_content=content, metadata=metadata))

print(f"Created {len(docs)} documents from individual records.")

2. Creating LangChain Documents...
  'combined_text' column not found in CSV. Generating rich semantic text using transform_emr_row_to_text...
Created 20630 documents from individual records.


In [4]:
print("3. Generating and Appending Summaries...")
model_summaries = get_model_summaries(df)
for model, text in model_summaries.items():
    docs.append(Document(page_content=text, metadata={"type": "model_summary", "machine_model": model}))

site_summaries = get_site_summaries(df)
for site, text in site_summaries.items():
    docs.append(Document(page_content=text, metadata={"type": "site_summary", "branch_site": site}))

print(f"Total documents including summaries: {len(docs)}.")

3. Generating and Appending Summaries...
[2026-06-14 12:54:21,920] INFO — src.transform — Generated 184 model summary documents.
[2026-06-14 12:54:21,956] INFO — src.transform — Generated 55 site summary documents.
Total documents including summaries: 20869.


In [5]:
print("4. Ingesting into Qdrant...")
embeddings = get_embeddings()
url = settings.qdrant_url
collection = settings.qdrant_collection

print(f"Connecting to Qdrant at {url}, collection: {collection}")

# Note: This creates a new collection or adds to an existing one
qdrant = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    url=url,
    prefer_grpc=False,
    collection_name=collection,
    force_recreate=True
)
print("Ingestion complete.")

4. Ingesting into Qdrant...
Connecting to Qdrant at http://localhost:6333, collection: emr_documents


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7416.09it/s]


Ingestion complete.


In [6]:
print("5. Testing Similarity Search...")
query = "Apa penyebab engine overheating?"
results = qdrant.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

5. Testing Similarity Search...

--- Result 1 ---
[EMR: U-00016365 | Model: D31PX-22 (KOMAT) | SN: 62102 | Site: PLB | Customer: NANDA SUNGAI MELINTANG]
Kategori Masalah: Engine Overheat Issues
Kejadian: TRS ENGINE OVERHEAT D31 62102 NSM
Gejala (Clean): Engine Overheating (Mentah: ENGINE OVERHEAT)
Penyebab (Clean): Kebocoran Radiator Upper Tank (Mentah: Caused of Problem : Engine overheat cause radiator clogging ?)
Tindakan Perbaikan (Clean): Cleaning Radiator Core
Tipe: Internal - Troubleshooting | PM: TRS
Tanggal: 10/20/2025 -> Closed: 11/25/2025

--- Result 2 ---
[EMR: U-00019722 | Model: PC135F-10M0 (KOMAT) | SN: J12757 | Site: SPR | Customer: KOP. SERBA USAHA JASA ABADI]
Kategori Masalah: Engine Overheat Issues
Kejadian: ENGINE COOLING TEMPERATURE ABNORMAL K12
Gejala (Clean): Low Coolant Level and Engine Overheating (Mentah: ENGINE COOLING TEMPERATURE ABNORMAL)
Penyebab (Clean): Penyebab Lain / Khusus (Mentah: Bugnet radiator ( Agak kotor,serangga ),)
Tindakan Perbaikan (Clean): I